# **Estimasi Cepat Sebaran Genangan Banjir**

Berbasis integrasi antara *Height Above Nearest Drainage* (HAND) dan data koordinat kedalaman genangan banjir lapangan.

Output:
1.   Raster kedalaman genangan banjir
2.   Poligon sebaran genangan banjir
3.   Data luas genangan banjir per wilayah administrasi

Created by: Syam'ani (https://github.com/syamaniulm) &copy; 2026

License: GNU General Public License v3.0 (https://www.gnu.org/licenses/gpl-3.0.html)

Sitasi:


Penggunaan HAND di dalam dokumen resmi wajib mengutip literatur ini: https://www.sciencedirect.com/science/article/abs/pii/S0022169411002599

# **Peringatan!**

Kode program ini merupakan proyek eksperimental. Sehingga masih memerlukan validasi di lapangan.

In [ ]:
!pip install -U geemap

In [ ]:
import ee, geemap, pandas as pd, geopandas as gpd

In [ ]:
ee.Authenticate()

In [ ]:
ee.Initialize(project='your-ee-project')      # Sesuaikan 'your-ee-project' dengan nama Google Earth Engine Project Anda

In [ ]:
# Mengakses Google Drive

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

path = '/content/drive/MyDrive'

In [ ]:
# Membuka file data koordinat kedalaman genangan banjir lapangan

flood_locations = pd.read_csv(path + '/flood/Flood_Locations.csv')      # Sesuaikan path dan nama filenya
flood_locations_gdf = gpd.GeoDataFrame(flood_locations, geometry=gpd.points_from_xy(flood_locations.Long, flood_locations.Lat), crs='EPSG:4326')
flood_locations_gdf.head()

In [ ]:
# Konversi tabel data banjir lapangan menjadi fitur titik

flood_points = geemap.geopandas_to_ee(flood_locations_gdf)
flood_points

In [ ]:
# Mengakses data catchment area dan HAND
# Catatan: Nilai HAND dalam desimeter

c_area = '50k'        # Pilih di antara '5k', '10k', '25k', '50k', '100k' atau '250k'

ca = ee.FeatureCollection('projects/ee-syamaniulm/assets/carea').filterBounds(flood_points).filter(ee.Filter.eq('id', 'ca' + c_area))
hand = ee.Image('projects/ee-syamaniulm/assets/hand').select('h' + c_area).rename('HAND')

In [ ]:
# Mengambil nilai HAND untuk setiap titik banjir lapangan

flood_location_hand_values = hand.sampleRegions(collection=flood_points, scale=30).aggregate_array('HAND').getInfo()
flood_location_hand_values  = [hand_values/10 if hand_values <= 250 else 0 for hand_values in flood_location_hand_values]

In [ ]:
# Menghitung total kedalaman genangan dari permukaan sungai terdekat
# Total kedalaman genangan = kedalaman banjir lapangan + HAND

flood_locations['Hand'] = flood_location_hand_values
flood_locations['Total_Depth'] = flood_locations['Depth'] + flood_locations['Hand']
flood_locations.head()

In [ ]:
# Analisis statistik untuk data total kedalaman genangan yang tidak wajar (anomali)
# menggunakan metode Interquartile Range (IQR)

import numpy as np

total_depth = np.array(flood_locations['Total_Depth'].to_list())

q1 = np.percentile(total_depth, 25)
q3 = np.percentile(total_depth, 75)

iqr = q3 - q1

upper_bound = (q3 + 1.5 * iqr).item()
upper_bound

In [ ]:
# Jika diperlukan, buang anomali data total kedalaman genangan

flood_location_filtered = flood_locations.drop(flood_locations[flood_locations['Hand'] == 0].index)
flood_location_filtered = flood_location_filtered.drop(flood_location_filtered[flood_location_filtered['Total_Depth'] > upper_bound].index)   # Komentari baris ini jika semua data dipakai (tidak ada anomali yang dibuang)
flood_location_filtered.head()

In [ ]:
# Konversi data total kedalaman genangan menjadi fitur titik

flood_location_filtered_gdf = gpd.GeoDataFrame(flood_location_filtered, geometry=gpd.points_from_xy(flood_location_filtered.Long, flood_location_filtered.Lat), crs='EPSG:4326')
flood_location_filtered_feature = geemap.geopandas_to_ee(flood_location_filtered_gdf)
flood_location_filtered_feature

In [ ]:
# Interseksi antara catchment area dan fitur titik total kedalaman genangan
# untuk mendapatkan total kedalaman genangan maksimum per catchment area

def max_depth_per_ca(feature):
    ca_geom = feature.geometry()
    intersected = flood_location_filtered_feature.filterBounds(ca_geom)
    max_depth = ee.List(intersected.aggregate_array('Total_Depth')).reduce(ee.Reducer.max())
    return feature.set('Max_Total_Depth', max_depth)

ca_with_max_depth = ca.map(max_depth_per_ca)
ca_with_max_depth

In [ ]:
# Kalibrasi nilai HAND dari satuan desimeter ke meter

hand_flood = hand.updateMask(hand.lte(250)).divide(10)

In [ ]:
# Konversi catchment area menjadi total kedalaman genangan maksimum

hand_ref = hand_flood.select('HAND').projection()

max_depth_image = (
    ca_with_max_depth
    .reduceToImage(properties=['Max_Total_Depth'], reducer=ee.Reducer.first())
    .rename('Max_Total_Depth')
    .reproject(crs=hand_ref.crs(), scale=hand_ref.nominalScale())
)

max_depth_image

In [ ]:
# Ekstraksi data kedalaman genangan banjir
# Kedalaman genangan = Totak kedalaman genangan maksimum - HAND

flood_depth = max_depth_image.subtract(hand_flood).rename('Flood_Depth')
flood_depth = flood_depth.updateMask(flood_depth.gt(0)).updateMask(hand_flood.gt(0))

In [ ]:
# Ekstraksi poligon sebaran genangan banjir

flooded_area = flood_depth.gt(0).selfMask().reduceToVectors(
    scale=flood_depth.projection().nominalScale(),
    geometry=flood_depth.geometry(),
    geometryType='polygon',
    eightConnected=True,
    labelProperty='mask',
    maxPixels=1e13
)

flooded_area

In [ ]:
# Jika diperlukan, data kedalaman genangan banjir dapat diekspor ke GeoTIFF
# File akan tersimpan di dalam Google Drive
# Catatan: Ekspor bisa gagal jika wilayah genangan banjir cukup luas (ukuran file terlalu besar)
# Kunjungi laman https://geemap.org/notebooks/118_download_image/ untuk referensi ekspor file raster yang cukup besar

output_geotiff_path = path + '/flood/flood_depth.tif'  # Sesuaikan path dan nama filenya
geemap.ee_export_image(flood_depth, filename=output_geotiff_path, scale=flood_depth.projection().nominalScale(), region=flooded_area.geometry(), file_per_band=False)

In [ ]:
# Ekstraksi data kedalaman banjir maksimum hasil estimasi

import math

max_flood_depth = math.floor(geemap.image_stats(flood_depth, region=ca.geometry(), scale=30).getInfo()['max']['Flood_Depth'])
max_flood_depth

In [ ]:
# Mengakses fitur data administrasi
# Catatan: Fitur data administrasi bersumber dari Badan Informasi Geospasial (BIG) (https://tanahair.indonesia.go.id/portal-web/)

admin = ee.FeatureCollection('projects/ee-syamaniulm/assets/admin').filterBounds(flooded_area)

In [ ]:
# Visualisasi sebaran dan kedalaman genangan banjir hasil estimasi

flood_vis = {'min': 0, 'max': max_flood_depth, 'palette': 'Blues'}
flood_style = {'color': 'blue', 'fillColor': '00000000', 'width': 1}
admin_style = {'color': 'red', 'fillColor': '00000000', 'width': 1}

Map = geemap.Map(basemap='HYBRID')
Map.centerObject(flooded_area, 12)
Map.addLayer(flood_depth, flood_vis, 'Flood Depth', opacity=0.7)
Map.addLayer(flood_location_filtered_feature, {'color': 'red'}, 'Flood Locations', shown=False)
Map.addLayer(flooded_area.style(**flood_style), {}, 'Flooded Area', shown=False)
Map.addLayer(admin.style(**admin_style), {}, 'Administrative Boundaries', shown=False)
# Map.add_labels(data=admin, column='NAMOBJ', font_size='10pt', font_color='yellow', layer_name='Desa/Kelurahan')
Map.add_colorbar(flood_vis,label='Flood Depth (m)', layer_name='Flood Depth', orientation='horizontal')
Map

In [ ]:
# Konversi fitur data administrasi dan poligon batas administrasi menjadi GeoDataFrame

admin_gdf = geemap.ee_to_gdf(admin)
flooded_area_gdf = geemap.ee_to_gdf(flooded_area).dissolve(by='mask')

In [ ]:
# Jika diperlukan, poligon sebaran genangan banjir dapat diekspor menjadi shapefile
# Shapefile akan tersimpan ke dalam Google Drive

output_shp_path = path + '/flood/flooded_area.shp'      # Sesuaikan path dan nama filenya
flooded_area_gdf.to_file(output_shp_path, driver='ESRI Shapefile')

In [ ]:
# Tumpang susun (intersect) antara poligon administrasi dan poligon sebaran genangan banjir

admin_flooded_area_gdf = gpd.overlay(admin_gdf, flooded_area_gdf, how='intersection')

In [ ]:
# Kalkulasi luas wilayah tergenang banjir untuk setiap wilayah administrasi

admin_flooded_area_gdf_proj = admin_flooded_area_gdf. to_crs('EPSG:32750') # Reproject to UTM Zone 50S
admin_flooded_area_gdf_proj['AREA_HA'] = round(admin_flooded_area_gdf_proj.geometry.area / 10000, 2) # Convert m^2 to hectares
admin_flooded_area_gdf_proj = admin_flooded_area_gdf_proj[['WADMPR','WADMKK','WADMKC','WADMKD', 'AREA_HA']].rename(columns={'WADMPR': 'Provinsi', 'WADMKK': 'Kabupaten/Kota', 'WADMKC': 'Kecamatan', 'WADMKD': 'Desa/Kelurahan', 'AREA_HA': 'Luas Genangan (hektare)'})

In [ ]:
# Ekstraksi dan visualisasi luas genangan banjir per desa/kelurahan

pd.set_option('display.max_rows', None)
admin_flooded_area_gdf_proj

In [ ]:
# Jika diperlukan, tabel luas area tergenang banjir per desa/kelurahan dapat diekspor ke file Excel
# File Excel akan tersimpan di dalam Google Drive

output_excel_path = path + '/flood/admin_flooded_area_summary.xlsx'  # Sesuaikan path dan nama filenya
admin_flooded_area_gdf_proj.to_excel(output_excel_path, index=False)

In [ ]:
# Ekstraksi dan visualisasi luas genangan banjir per kecamatan

kecamatan_summary = admin_flooded_area_gdf_proj.groupby('Kecamatan')['Luas Genangan (hektare)'].sum().reset_index()
kecamatan_summary

In [ ]:
# Ekstraksi dan visualisasi luas genangan banjir per kabupaten/kota

kabupaten_summary = admin_flooded_area_gdf_proj.groupby('Kabupaten/Kota')['Luas Genangan (hektare)'].sum().reset_index()
kabupaten_summary